In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
import sqlite3

BASE = '/content/drive/MyDrive/Akhila-portfolio/Project1-data/'
print('folder ready')


folder ready


In [8]:
bene_cols = ['DESYNPUF_ID','BENE_BIRTH_DT','BENE_SEX_IDENT_CD',
 'BENE_RACE_CD','SP_ALZHDMTA','SP_CHF','SP_CHRNKIDN','SP_CNCR',
 'SP_COPD','SP_DEPRESSN','SP_DIABETES','SP_ISCHMCHT',
 'SP_OSTEOPRS','SP_RA_OA','SP_STRKETIA',
 'MEDREIMB_IP','MEDREIMB_OP','MEDREIMB_CAR']
bene = pd.read_csv(BASE + 'DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv',
                   usecols=bene_cols, dtype={'DESYNPUF_ID': str})
print(bene.shape)
bene.head()


(116352, 18)


,DESYNPUF_ID,BENE_BIRTH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,SP_ALZHDMTA,SP_CHF,SP_CHRNKIDN,SP_CNCR,SP_COPD,SP_DEPRESSN,SP_DIABETES,SP_ISCHMCHT,SP_OSTEOPRS,SP_RA_OA,SP_STRKETIA,MEDREIMB_IP,MEDREIMB_OP,MEDREIMB_CAR
0,00013D2EFD8E45D1,19230501,1,1,2,2,2,2,2,2,2,2,2,2,2,0.0,50.0,0.0
1,00016F745862898F,19430101,1,1,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,700.0
2,0001FDD721E223DC,19360901,2,1,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0
3,00021CA6FF03E670,19410601,1,5,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0
4,00024B3D2352D2D0,19360801,1,1,2,2,2,2,2,2,2,2,1,2,2,0.0,30.0,220.0


In [9]:
ip_cols = ['DESYNPUF_ID','CLM_ID','CLM_FROM_DT','CLM_THRU_DT',
 'PRVDR_NUM','CLM_PMT_AMT','CLM_DRG_CD','ICD9_DGNS_CD_1','HCPCS_CD_1']
op_cols = ['DESYNPUF_ID','CLM_ID','CLM_FROM_DT','CLM_THRU_DT',
 'PRVDR_NUM','CLM_PMT_AMT','ICD9_DGNS_CD_1','HCPCS_CD_1']
ip = pd.read_csv(BASE + 'DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv',
                 usecols=ip_cols, dtype={'DESYNPUF_ID': str, 'CLM_ID': str})
op = pd.read_csv(BASE + 'DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv',
                 usecols=op_cols, dtype={'DESYNPUF_ID': str, 'CLM_ID': str})
print('inpatient rows:', len(ip))
print('outpatient rows:', len(op))


inpatient rows: 66773
outpatient rows: 790790


In [10]:
line_pmt = [f'LINE_NCH_PMT_AMT_{i}' for i in range(1, 14)]
car_cols = ['DESYNPUF_ID','CLM_ID','CLM_FROM_DT','CLM_THRU_DT',
 'ICD9_DGNS_CD_1','HCPCS_CD_1','PRF_PHYSN_NPI_1'] + line_pmt
car_a = pd.read_csv(BASE + 'DE1_0_2008_to_2010_Carrier_Claims_Sample_1A.csv',
                    usecols=car_cols, dtype={'DESYNPUF_ID': str, 'CLM_ID': str})
car_b = pd.read_csv(BASE + 'DE1_0_2008_to_2010_Carrier_Claims_Sample_1B.csv',
                    usecols=car_cols, dtype={'DESYNPUF_ID': str, 'CLM_ID': str})
carrier = pd.concat([car_a, car_b], ignore_index=True)
carrier['paid_amt'] = carrier[line_pmt].fillna(0).sum(axis=1)
print('carrier rows:', len(carrier))
print('carrier total paid:', carrier['paid_amt'].sum())


carrier rows: 4741335
carrier total paid: 414603690.0


In [11]:
pde = pd.read_csv(BASE + 'DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv',
                  dtype={'DESYNPUF_ID': str})
def simple(df, ctype, paid_col, provider_col):
    cols = ['DESYNPUF_ID','CLM_ID','CLM_THRU_DT', paid_col,
            'ICD9_DGNS_CD_1','HCPCS_CD_1', provider_col]
    out = df[cols].copy()
    out.columns = ['patient_id','claim_id','thru_dt','paid_amt',
                   'diag1','hcpcs1','provider_id']
    out['claim_type'] = ctype
    return out
all_claims = pd.concat([
    simple(ip, 'Inpatient', 'CLM_PMT_AMT', 'PRVDR_NUM'),
    simple(op, 'Outpatient', 'CLM_PMT_AMT', 'PRVDR_NUM'),
    simple(carrier, 'Carrier', 'paid_amt', 'PRF_PHYSN_NPI_1')],
    ignore_index=True)
conn = sqlite3.connect(BASE + 'medicare.db')
bene.to_sql('beneficiary', conn, if_exists='replace', index=False)
ip.to_sql('inpatient', conn, if_exists='replace', index=False)
op.to_sql('outpatient', conn, if_exists='replace', index=False)
carrier.to_sql('carrier', conn, if_exists='replace', index=False)
pde.to_sql('pde', conn, if_exists='replace', index=False)
all_claims.to_sql('all_claims', conn, if_exists='replace', index=False)
conn.close()
print('all_claims rows:', len(all_claims))
print('database saved')


/tmp/ipykernel_521/2874296136.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  pde = pd.read_csv(BASE + 'DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv',


all_claims rows: 5598898
database saved


In [12]:
print(all_claims.groupby('claim_type')['paid_amt'].agg(['count','sum','mean']))
print('patients:', bene['DESYNPUF_ID'].nunique())
print('zero-paid claims:', (all_claims['paid_amt'] == 0).sum())


              count          sum         mean
claim_type                                   
Carrier     4741335  414603690.0    87.444505
Inpatient     66773  639260180.0  9573.632756
Outpatient   790790  224524710.0   283.924569
patients: 116352
zero-paid claims: 511831


In [13]:
print('pde rows:', len(pde))


pde rows: 5552421
